In [ ]:
from pathlib import Path
import json
import pickle
import sys

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from skimage.segmentation import find_boundaries

PROJECT_DIR = Path("/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC")
SPATIALPYALAB_DIR = Path("/Volumes/B/Projects/2025Project/SpatialAnalysis/Code")
if str(SPATIALPYALAB_DIR) not in sys.path:
    sys.path.append(str(SPATIALPYALAB_DIR))
import SpatialPyALab as lab

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
})


In [ ]:
TUMOR_GRAY = "#808080"
CELLTYPE_COLORS = {
    "Bcell": "#BA55D3",
    "Tcell": "#FF0000",
    "Myeloid": "#FFD500",
    "Endothelial": "#1E90FF",
    "Fibroblast": "#BD8F8F",
    "Epithelial": TUMOR_GRAY,
    "Hepatocyte": "#1E8157",
    "Hepatocytes": "#1E8157",
}
SEP_COLORS = {
    "N_SEP1": "#bf9191", "N_SEP2": "#FFBE0B", "N_SEP3": "#78A978",
    "N_SEP4": "#FF7944", "N_SEP5": "#BA55D3", "N_SEP6": "#A8DADC",
    "N_SEP7": "#ACE894", "N_SEP8": "#FF78B3", "N_SEP9": "#FF0000",
    "N_SEP10": "#2C6E49", "N_SEP11": "#57A8FA", "N_SEP12": "#BC5C69",
}
SEP_COLORS.update({f"M_SEP{i}": TUMOR_GRAY for i in range(1, 6)})
CELLTYPE_GROUPS = {
    "Immune": ["Bcell", "Tcell", "Myeloid", "Epithelial"],
    "Stromal": ["Endothelial", "Fibroblast", "Epithelial"],
}
CELLTYPE_TITLES = {
    "Immune": "B cell + T cell + Myeloid + Epithelial",
    "Stromal": "Endothelial + Fibroblast + Epithelial",
}
SEP_SHOW = ["N_SEP1", "N_SEP11", "N_SEP12", "N_SEP4", "N_SEP6", "N_SEP8"]
SEP_FILE = PROJECT_DIR / "Data/ST/SEP/All_hex_MP_scores_for_h5ad.csv"

COMMON_CONFIG = {
    "layer_width": 20,
    "max_outward_distance": 100,
    "image_key": "staining_image",
    "mask_key": "staining_image_mask",
    "cell_id_col": "cell_id",
    "mask_id_col": "cell_ID_mask",
    "celltype_col": "CellType",
    "background_alpha": 0.45,
    "show_figures": True,
    "sep_file": SEP_FILE,
    "celltype_colors": CELLTYPE_COLORS,
    "sep_colors": SEP_COLORS,
    "celltype_groups": CELLTYPE_GROUPS,
    "celltype_titles": CELLTYPE_TITLES,
    "sep_show": SEP_SHOW,
}
SAMPLE_CONFIGS = {
    "Pt5-4": {
        **COMMON_CONFIG,
        "sample_id": "SR250530M3",
        "root_dir": PROJECT_DIR / "Data/ST/Samples/SR250530M3",
        "region_k": 3,
        "crop": (6700, 9200, 9300, 12000),
        "fill_alpha": 0.95,
        "outdir": PROJECT_DIR / "Figure/ST/Boundary/Figure5D",
    },
    "Pt6-5": {
        **COMMON_CONFIG,
        "sample_id": "SR250521M1",
        "root_dir": PROJECT_DIR / "Data/ST/Samples/SR250521M1",
        "region_k": 1,
        "crop": (6200, 8500, 6000, 8200),
        "fill_alpha": 0.85,
        "outdir": PROJECT_DIR / "Figure/ST/Boundary",
    },
}
for config in SAMPLE_CONFIGS.values():
    config.update({
        "cell_file": config["root_dir"] / "cell_segment/Cell_Segment_addCelltype_adata.h5ad",
        "integrated_cell_file": config["root_dir"] / "integrated/cell_adata.h5ad",
        "polygon_file": config["root_dir"] / "cell_segment/Region_Clone_Polys_by_GLMPCA.pkl",
        "clone_file": config["root_dir"] / "cell_segment/region_to_clone.json",
    })
del config
SAMPLES_TO_RUN = list(SAMPLE_CONFIGS)


In [ ]:
def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required input file does not exist: {path}")
    return path


def require_columns(frame, columns, label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{label} is missing columns: {missing}")


def checked_ids(frame, column, label):
    require_columns(frame, [column], label)
    ids = frame[column].astype("string")
    if ids.isna().any() or ids.str.strip().eq("").any():
        raise ValueError(f"{label}.{column} contains missing or empty IDs")
    if ids.duplicated().any():
        raise ValueError(f"{label}.{column} must contain unique IDs")
    return ids


def load_cell_adata(config):
    adata = sc.read_h5ad(require_file(config["cell_file"]))
    require_columns(
        adata.obs,
        [config["cell_id_col"], config["mask_id_col"], config["celltype_col"]],
        "cell.obs",
    )
    adata = adata[adata.obs[config["celltype_col"]].astype(str).ne("conflict")].copy()
    checked_ids(adata.obs, config["cell_id_col"], "cell.obs")
    for key in [config["image_key"], config["mask_key"]]:
        if key not in adata.uns:
            raise KeyError(f"cell.uns is missing {key!r}")
    if "spatial" not in adata.obsm:
        raise KeyError("cell.obsm is missing 'spatial' image coordinates")
    if np.asarray(adata.obsm["spatial"]).shape != (adata.n_obs, 2):
        raise ValueError("cell.obsm['spatial'] must have shape (n_cells, 2)")
    return adata


def prepare_spatial_um(adata):
    """Retain the original relation: spatial_um = spatial / mu_scale."""
    if "mu_scale" in adata.uns and "spatial_manual_fine" in adata.uns["mu_scale"]:
        mu_scale = np.asarray(adata.uns["mu_scale"]["spatial_manual_fine"], dtype=float)
        if mu_scale.ndim == 0 or mu_scale.size == 1:
            value = float(mu_scale.ravel()[0])
            mu_scale = np.array([value, value])
        else:
            mu_scale = mu_scale.ravel()[:2]
        if not np.isfinite(mu_scale).all() or (mu_scale <= 0).any():
            raise ValueError("The image-to-micrometer scale must be positive and finite")
        adata.obsm["spatial_um"] = np.asarray(adata.obsm["spatial"], dtype=float) / mu_scale
    else:
        if "spatial_um" not in adata.obsm:
            raise KeyError("cell data requires spatial_um or mu_scale['spatial_manual_fine']")
        if np.asarray(adata.obsm["spatial_um"]).shape != (adata.n_obs, 2):
            raise ValueError("cell.obsm['spatial_um'] must have shape (n_cells, 2)")
        with np.errstate(divide="ignore", invalid="ignore"):
            mu_scale = np.nanmedian(
                np.asarray(adata.obsm["spatial"], dtype=float)
                / np.asarray(adata.obsm["spatial_um"], dtype=float),
                axis=0,
            )
        mu_scale = np.asarray(mu_scale, dtype=float).ravel()[:2]
        if not np.isfinite(mu_scale).all() or (mu_scale <= 0).any():
            raise ValueError("Cannot estimate a positive, finite image-to-micrometer scale")
    return adata, mu_scale


def add_hex_sep(cell, config):
    cell_id_col = config["cell_id_col"]
    ref = sc.read_h5ad(require_file(config["integrated_cell_file"]), backed="r")
    try:
        require_columns(
            ref.obs,
            ["sample_id", cell_id_col, "matched_hex15_id", "hex15_match_accepted", config["mask_id_col"]],
            "integrated.obs",
        )
        obs = ref.obs.loc[ref.obs["sample_id"].astype(str).eq(config["sample_id"])].copy()
    finally:
        ref.file.close()
    if obs.empty:
        raise ValueError(f"No integrated cells found for {config['sample_id']}")

    target_ids = checked_ids(cell.obs, cell_id_col, "cell.obs")
    ref_ids = checked_ids(obs, cell_id_col, "integrated.obs for the sample")
    found = pd.Index(ref_ids).get_indexer(pd.Index(target_ids)) >= 0
    aligned = obs.set_index(pd.Index(ref_ids)).reindex(pd.Index(target_ids))
    accepted = (
        aligned["hex15_match_accepted"].astype("string").str.lower()
        .isin(["true", "1", "1.0"]).to_numpy()
    )
    accepted &= found
    hex_ids = aligned["matched_hex15_id"].astype("string").reset_index(drop=True)
    hex_ids.loc[~accepted] = pd.NA

    sep = pd.read_csv(
        require_file(config["sep_file"]),
        dtype={"SpotID": "string", "sample_id": "string"},
    )
    require_columns(sep, ["sample_id", "SpotID", "SEP"], "SEP CSV")
    sep = sep.loc[sep["sample_id"].eq(config["sample_id"])].copy()
    if sep.empty:
        raise ValueError(f"No SEP rows found for {config['sample_id']}")
    sep.index = pd.Index(checked_ids(sep, "SpotID", "SEP CSV for the sample"))
    attached = sep.reindex(pd.Index(hex_ids))

    cell.obs["matched_hex15_id"] = hex_ids.to_numpy()
    cell.obs["hex15_match_accepted"] = accepted
    if "dist_to_hex15" in aligned:
        cell.obs["dist_to_hex15"] = aligned["dist_to_hex15"].to_numpy()
    fields = {
        "SEP": "SEP_hex15", "SEP_score": "SEP_hex15_score",
        "SEP_second": "SEP_hex15_second", "SEP_second_score": "SEP_hex15_second_score",
        "SEP_margin": "SEP_hex15_margin",
    }
    for source, destination in fields.items():
        if source in attached:
            cell.obs[destination] = attached[source].to_numpy()
    labels = cell.obs["SEP_hex15"].astype("string").str.strip()
    missing_labels = ["", "nan", "none", "na", "unassigned", "ua", "<na>"]
    labels = labels.mask(labels.str.lower().isin(missing_labels))
    cell.obs["SEP_hex15"] = labels.to_numpy()
    return cell


def load_region_files(config):
    with require_file(config["polygon_file"]).open("rb") as handle:
        polygons = pickle.load(handle)
    clone_file = Path(config["clone_file"])
    if clone_file.is_file():
        with clone_file.open("r", encoding="utf-8") as handle:
            region_to_clone = json.load(handle)
    else:
        region_to_clone = {}
        print(f"Clone labels are unavailable; clone will be missing: {clone_file}")
    if not isinstance(region_to_clone, dict):
        raise TypeError("The clone-label JSON must contain an object mapping region IDs to labels")
    return polygons, region_to_clone


In [ ]:
def make_layers(polygons, config):
    width = config["layer_width"]
    distance = config["max_outward_distance"]
    if width <= 0 or distance < 0 or distance % width != 0:
        raise ValueError("Outward distance must be a nonnegative multiple of the positive layer width")
    return lab.by.create_layers(
        polys=polygons,
        operation_mode="per_poly",
        expansion_mode="both",
        buffer_step=width,
        num_steps=distance // width,
        min_area=20,
        consider_others_expansion=True,
        safe_margin=0.0,
        include_center=True,
        sliding_mode=False,
        window_size=None,
        slide_step=None,
        max_dist=None,
    )


def select_target_layers(layers, config):
    region_k = config["region_k"]
    if region_k not in layers:
        raise KeyError(f"Region {region_k} is unavailable; available keys: {list(layers)}")
    available = set()
    for _, number in layers[region_k]:
        try:
            value = float(number)
        except (TypeError, ValueError):
            continue
        if np.isfinite(value):
            available.add(value)
    selected = sorted(
        value for value in available
        if value <= 0 or value * config["layer_width"] <= config["max_outward_distance"]
    )
    if not selected:
        raise ValueError(f"No layers are available for region {region_k}")
    return selected


def assign_layers_to_cells(adata, layers, region_to_clone, config):
    cell_id_col = config["cell_id_col"]
    layer_matrix, assignments = lab.by.assign_points_nonoverlap_fast_from_obsm(
        adata=adata,
        layers_dict=layers,
        obsm_key="spatial_um",
        cell_id_col=cell_id_col,
        swap_xy=False,
    )
    del layer_matrix
    require_columns(assignments, [cell_id_col, "region_key", "layer"], "layer assignments")
    assignments = assignments.copy()
    assignments[cell_id_col] = assignments[cell_id_col].astype(str)
    assignments["clone"] = assignments["region_key"].map(region_to_clone)
    assignments = assignments.drop_duplicates(subset=cell_id_col, keep="first")

    layer_cell = adata.copy()
    stale = [column for column in ["region_key", "layer", "distance", "clone"] if column in layer_cell.obs]
    obs = layer_cell.obs.drop(columns=stale).copy()
    original_index = obs.index.copy()
    obs[cell_id_col] = obs[cell_id_col].astype(str)
    obs["_layer_obs_order"] = np.arange(len(obs))
    merged = obs.merge(assignments, on=cell_id_col, how="left", sort=False, validate="one_to_one")
    merged = merged.sort_values("_layer_obs_order").drop(columns="_layer_obs_order")
    merged.index = original_index
    merged["distance"] = pd.to_numeric(merged["layer"], errors="coerce") * config["layer_width"]
    layer_cell.obs = merged
    return layer_cell


def crop_image_and_mask(adata, config):
    x1, x2, y1, y2 = config["crop"]
    image = np.asarray(adata.uns[config["image_key"]])
    mask = np.asarray(adata.uns[config["mask_key"]])
    if image.ndim not in (2, 3) or mask.ndim != 2 or image.shape[:2] != mask.shape:
        raise ValueError("Staining image and segmentation mask must have matching first two dimensions")
    if not (0 <= x1 < x2 <= image.shape[0] and 0 <= y1 < y2 <= image.shape[1]):
        raise ValueError(f"Crop {config['crop']} is outside the image shape {image.shape[:2]}")
    image_crop = image[x1:x2, y1:y2].copy()
    image_display = image_crop.transpose(1, 0, 2) if image_crop.ndim == 3 else image_crop.T
    mask_display = mask[x1:x2, y1:y2].copy().T
    return image_display, mask_display


def filter_cells_for_plot(layer_cell, scale, target_layers, config):
    x1, x2, y1, y2 = config["crop"]
    obs = layer_cell.obs.copy()
    obs["_layer_num"] = pd.to_numeric(obs["layer"], errors="coerce")
    obs["_cell_id_mask_num"] = pd.to_numeric(obs[config["mask_id_col"]], errors="coerce")
    region = obs["region_key"].astype(str)
    region_mask = region.eq(f"region_{config['region_k']}") | region.eq(str(config["region_k"]))
    layer_mask = obs["_layer_num"].isin(target_layers)
    pixels = np.asarray(layer_cell.obsm["spatial_um"], dtype=float) * scale
    crop_mask = (
        (pixels[:, 0] >= x1) & (pixels[:, 0] < x2)
        & (pixels[:, 1] >= y1) & (pixels[:, 1] < y2)
    )
    return obs.loc[region_mask.to_numpy() & layer_mask.to_numpy() & crop_mask].copy()


In [ ]:
def make_all_cell_boundary_rgba(mask_display, color=(0.05, 0.05, 0.05, 0.55)):
    boundaries = find_boundaries(mask_display.astype(np.int64), mode="inner")
    rgba = np.zeros((*mask_display.shape, 4), dtype=np.float32)
    rgba[boundaries] = color
    return rgba


def make_celltype_pixel_rgba(mask_display, obs, labels, annotation, colors, alpha):
    rgba = np.zeros((*mask_display.shape, 4), dtype=np.float32)
    for label in labels:
        mask_ids = (
            obs.loc[obs[annotation].astype(str).eq(label), "_cell_id_mask_num"]
            .dropna().astype(int).unique()
        )
        if len(mask_ids) == 0:
            continue
        pixels = np.isin(mask_display, mask_ids)
        raw_color = colors.get(label, "#00FF00")
        if isinstance(raw_color, str):
            color = to_rgba(raw_color, alpha=alpha)
        else:
            raw_color = tuple(raw_color)
            if len(raw_color) == 4:
                color = raw_color
            elif len(raw_color) == 3:
                color = (*raw_color, alpha)
            else:
                raise ValueError(f"Invalid color for {label}: {raw_color}")
        rgba[pixels] = color
    return rgba


def iter_polygons(geometry):
    if geometry is None or getattr(geometry, "is_empty", False):
        return
    if geometry.geom_type == "Polygon":
        yield geometry
    elif geometry.geom_type in ("MultiPolygon", "GeometryCollection"):
        for item in geometry.geoms:
            yield from iter_polygons(item)


def plot_polygon_boundary_on_crop(ax, geometry, scale, crop, linewidth, alpha, zorder):
    x1, _, y1, _ = crop
    for polygon in iter_polygons(geometry):
        for ring in [polygon.exterior, *polygon.interiors]:
            pixels = np.asarray(ring.coords, dtype=float) * scale
            ax.plot(
                pixels[:, 0] - x1,
                pixels[:, 1] - y1,
                color="black",
                linewidth=linewidth,
                alpha=alpha,
                linestyle="-",
                zorder=zorder,
            )


def plot_region_layers(ax, layers, target_layers, scale, config):
    for geometry, number in layers[config["region_k"]]:
        try:
            number = float(number)
        except (TypeError, ValueError):
            continue
        if number not in target_layers:
            continue
        if number == 0:
            linewidth, alpha, zorder = 1.8, 1.0, 6
        elif abs(number) == 1:
            linewidth, alpha, zorder = 0.9, 0.95, 5
        else:
            linewidth, alpha, zorder = 0.6, 0.90, 5
        plot_polygon_boundary_on_crop(
            ax, geometry, scale, config["crop"], linewidth, alpha, zorder,
        )


def paint_panel(
    ax, image, mask, obs, annotation, colors, selection, boundaries,
    layers, scale, target_layers, panel_title, config,
):
    is_sep = annotation == "SEP_hex15"
    palette = dict(colors)
    present = obs[annotation].dropna().astype(str).unique().tolist()
    if selection is None:
        labels = [label for label in palette if label in present]
        labels += sorted(set(present) - set(palette))
    else:
        labels = [label for label in selection if label in present]
    unknown = [label for label in labels if label not in palette]
    for index, label in enumerate(unknown):
        palette[label] = plt.get_cmap("tab20")(index % 20)

    rgba = make_celltype_pixel_rgba(
        mask, obs, labels, annotation, palette, alpha=config["fill_alpha"],
    )
    ax.imshow(image, alpha=config["background_alpha"], zorder=0)
    ax.imshow(rgba, zorder=2)
    ax.imshow(boundaries, zorder=4)
    plot_region_layers(ax, layers, target_layers, scale, config)
    ax.set_xlim(0, image.shape[1])
    ax.set_ylim(image.shape[0], 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(
        f"{config['sample_id']} | Region {config['region_k']} | {panel_title}",
        fontsize=10,
    )
    handles = [Patch(facecolor=palette[label], label=label.replace("_", "-")) for label in labels]
    if handles:
        ax.legend(
            handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1),
            frameon=False, fontsize=8, borderaxespad=0,
        )
    if is_sep:
        missing = obs["SEP_hex15"].isna().sum()
        ax.text(
            0, -0.04,
            f"Only selected N-SEP states are shown; no SEP: {missing}/{len(obs)} cells",
            transform=ax.transAxes, fontsize=8, va="top",
        )


def save_figure(figure, stem, config):
    outdir = Path(config["outdir"])
    outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / f"{stem}.pdf"
    try:
        with plt.rc_context({"image.composite_image": False}):
            figure.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        if config["show_figures"]:
            plt.show()
    finally:
        plt.close(figure)
    return path


In [ ]:
def main(config):
    cell = load_cell_adata(config)
    cell, scale = prepare_spatial_um(cell)
    cell = add_hex_sep(cell, config)
    polygons, region_to_clone = load_region_files(config)
    layers = make_layers(polygons, config)
    target_layers = select_target_layers(layers, config)
    layer_cell = assign_layers_to_cells(cell, layers, region_to_clone, config)
    image, mask = crop_image_and_mask(cell, config)
    obs = filter_cells_for_plot(layer_cell, scale, target_layers, config)
    boundaries = make_all_cell_boundary_rgba(mask)

    x1, x2, y1, y2 = config["crop"]
    prefix = f"{config['sample_id']}_Region{config['region_k']}_x{x1}-{x2}_y{y1}-{y2}"
    panels = [
        (config["celltype_col"], config["celltype_colors"], labels, config["celltype_titles"][name])
        for name, labels in config["celltype_groups"].items()
    ]
    panels.append(("SEP_hex15", config["sep_colors"], config["sep_show"], "N-SEP1, 11, 12, 4, 6, 8 only"))
    count = len(panels)
    figure = plt.figure(figsize=(12 * count, 8), dpi=200)
    try:
        for index, (annotation, colors, selection, title) in enumerate(panels):
            ax = figure.add_axes([(index + 0.04) / count, 0.10, 0.66 / count, 0.80])
            paint_panel(
                ax, image, mask, obs, annotation, colors, selection, boundaries,
                layers, scale, target_layers, title, config,
            )
        figure_path = save_figure(figure, f"{prefix}_Bcell_Stroma_SEP_comparison", config)
    finally:
        plt.close(figure)

    result_columns = [
        config["cell_id_col"], config["mask_id_col"], config["celltype_col"],
        "region_key", "layer", "distance", "clone", "matched_hex15_id",
        "hex15_match_accepted", "dist_to_hex15", "SEP_hex15", "SEP_hex15_score",
        "SEP_hex15_second", "SEP_hex15_second_score", "SEP_hex15_margin",
    ]
    result_columns = list(dict.fromkeys(column for column in result_columns if column in obs))
    print(f"{config['sample_id']}: plotted {len(obs)} cells; saved {figure_path}")
    return {
        "sample_id": config["sample_id"],
        "plot_obs": obs[result_columns].copy(),
        "target_layers": list(target_layers),
        "prefix": prefix,
        "figure_path": figure_path,
    }


In [ ]:
plot_results = {}
for sample_name in SAMPLES_TO_RUN:
    plot_results[sample_name] = main(SAMPLE_CONFIGS[sample_name])
